In [1]:
from pathlib import Path
import json

import numpy as np
import pandas as pd

SIGNALS_PATH = Path(
    "../data/processed/microservices_sample/"
    "microservices_trace_signals.csv"
)

FEATURES_PATH = Path(
    "../data/processed/microservices_sample/"
    "microservices_trace_features.csv"
)

In [2]:
signals = pd.read_csv(SIGNALS_PATH)

signals["tc_trace_id"].duplicated().sum()

np.int64(0)

In [3]:
base_columns = [
    "event_count",
    "info_count",
    "warn_count",
    "error_count",
    "has_stack_trace",
    "has_exception_term",
    "has_failed_term",
    "has_timeout_term",
    "has_error_term",
    "service_count",
]

features = signals[
    ["tc_trace_id", *base_columns]
].copy()

features["error_rate"] = (
    features["error_count"]
    / features["event_count"]
)

features["warn_rate"] = (
    features["warn_count"]
    / features["event_count"]
)

text_columns = [
    "has_exception_term",
    "has_failed_term",
    "has_timeout_term",
    "has_error_term",
]

features["text_signal_count"] = features[
    text_columns
].sum(axis=1)

features["has_any_signal"] = (
    (features["error_count"] > 0)
    | (features["warn_count"] > 0)
    | (features["has_stack_trace"] > 0)
    | (features["text_signal_count"] > 0)
).astype(int)

features.head()

,tc_trace_id,event_count,info_count,warn_count,error_count,has_stack_trace,has_exception_term,has_failed_term,has_timeout_term,has_error_term,service_count,error_rate,warn_rate,text_signal_count,has_any_signal
0,00010db4-c32c-44a4-b010-d31214566aa1,1,1,0,0,0,0,0,0,0,1,0.000000,0.000000,0,0
1,00021ea9-f57b-448b-85db-e11b9f0e8a5a,4,4,0,0,0,0,0,0,0,1,0.000000,0.000000,0,0
2,0003d331-0143-40f3-8ad6-5b134e1dc0ba,1,1,0,0,0,0,0,0,0,1,0.000000,0.000000,0,0
3,000b87a9-b903-4a9d-b442-4c5a5114a47a,1,0,1,0,0,0,0,0,0,1,0.000000,1.000000,0,1
4,000e9a48-85c5-4aea-8c77-74dceb191d50,3,1,1,1,0,0,0,0,0,2,0.333333,0.333333,0,1


In [4]:
features.describe().T

,count,mean,std,min,25%,50%,75%,max
event_count,19976.0,7.746095,119.900150,1.0,1.0,2.0,4.0,14376.0
info_count,19976.0,7.027583,119.022163,0.0,1.0,1.0,4.0,14376.0
warn_count,19976.0,0.535843,5.105673,0.0,0.0,0.0,0.0,400.0
error_count,19976.0,0.182669,0.607028,0.0,0.0,0.0,0.0,54.0
has_stack_trace,19976.0,0.010112,0.100052,0.0,0.0,0.0,0.0,1.0
has_exception_term,19976.0,0.002453,0.049468,0.0,0.0,0.0,0.0,1.0
has_failed_term,19976.0,0.000451,0.021222,0.0,0.0,0.0,0.0,1.0
has_timeout_term,19976.0,0.002353,0.048450,0.0,0.0,0.0,0.0,1.0
has_error_term,19976.0,0.013166,0.113987,0.0,0.0,0.0,0.0,1.0
service_count,19976.0,1.290649,1.584318,1.0,1.0,1.0,1.0,53.0


In [5]:
### Célula 5 — distribuição dos sinais

signal_summary = pd.DataFrame({
    "feature": [
        "error_count",
        "warn_count",
        "has_stack_trace",
        "text_signal_count",
        "service_count",
    ],
    "traces_with_value": [
        int(features["error_count"].gt(0).sum()),
        int(features["warn_count"].gt(0).sum()),
        int(features["has_stack_trace"].gt(0).sum()),
        int(features["text_signal_count"].gt(0).sum()),
        int(features["service_count"].gt(1).sum()),
    ],
})

signal_summary

,feature,traces_with_value
0,error_count,3219
1,warn_count,4251
2,has_stack_trace,202
3,text_signal_count,308
4,service_count,3032
